# 🧠 InterviewIQ AI — Resume Skill Extraction & Role Classification
### Notebook 02: Candidate Profiling Pipeline

This notebook trains a machine learning pipeline that:
1. Preprocesses raw unstructured resume text.
2. Extracts domain-specific technical skills.
3. Classifies candidate profiles into target career roles using a supervised **TF-IDF + Logistic Regression** model.
4. Saves the trained artifacts for direct backend inference.

---
#### Mathematical Formulation:
Given resume text $d$, TF-IDF vectorization maps text to feature vector $x \in \mathbb{R}^V$:
$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \log\left(\frac{1 + N}{1 + \text{DF}(t)}\right)$$
The multinomial logistic regression predicts role class $k$:
$$P(Y = k | x) = \frac{e^{w_k^T x + b_k}}{\sum_{j=1}^K e^{w_j^T x + b_j}}$$

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

data_path = '../data/resume_skills.csv'
models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

df = pd.read_csv(data_path)
print(f"Loaded {len(df)} resumes.")
df.head(2)

## 1. Text Preprocessing & Label Encoding

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_resume'] = df['resume_text'].apply(clean_text)

le = LabelEncoder()
df['role_label'] = le.fit_transform(df['role_category'])

print("Class mapping:")
for idx, label in enumerate(le.classes_):
    print(f"  {idx} -> {label}")

## 2. Train / Test Split (Stratified)
We split data into 80% training and 20% testing sets, stratified by role label to prevent class imbalance.

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['cleaned_resume'],
    df['role_label'],
    test_size=0.20,
    random_state=42,
    stratify=df['role_label']
)

print(f"Training set: {len(X_train_raw)} samples")
print(f"Testing set:  {len(X_test_raw)} samples")

## 3. TF-IDF Vectorization & Classifier Training

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=2500,
    ngram_range=(1, 2),
    stop_words='english',
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)

print(f"Vocabulary size: {len(vectorizer.vocabulary_)} n-grams")

clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"\nTest Set Accuracy: {acc * 100:.2f}%")

## 4. Detailed Classification Report & Error Analysis

In [ ]:
print("--- Detailed Classification Report ---")
print(classification_report(y_test, y_pred, target_names=le.classes_))

## 5. Exporting Model Artifacts for Backend Inference

In [ ]:
joblib.dump(clf, os.path.join(models_dir, 'skill_classifier.joblib'))
joblib.dump(vectorizer, os.path.join(models_dir, 'skill_vectorizer.joblib'))
joblib.dump(le, os.path.join(models_dir, 'skill_label_encoder.joblib'))

print("All resume classification artifacts exported successfully to ../models/")